# 影子策略回测 - 10年验证

回测时间范围：2014-01-01 至 2024-12-31（10年）

策略：
1. 主线策略：假弱高开 + 情绪开关
2. 观察线策略：二板策略

回测平台：Ricequant Notebook研究环境

**注意**：此notebook需要在Ricequant平台Notebook研究环境中运行，本地无法运行。

In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Ricequant Notebook研究环境API会在平台上自动注入
# 本地运行需要安装 rqdatac: pip install rqdatac
# 并初始化: rqdatac.init('username', 'password')

# 如果在本地运行，尝试导入rqdatac
try:
    import rqdatac
    rqdatac.init()
    from rqdatac import all_instruments, history_bars, get_trading_dates, get_factor, index_components
    print("rqdatac导入成功（本地模式）")
except ImportError:
    print("警告: 未安装rqdatac")
    print("此notebook请在Ricequant Notebook研究环境中运行")
    print("或本地安装: pip install rqdatac 并配置账号")

plt.style.use('seaborn-v0_8' if 'seaborn-v0_8' in plt.style.available else 'seaborn')
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("库导入成功！")

## 1. 主线策略：假弱高开

In [ ]:
class MainlineStrategy:
    """
    主线策略：假弱高开 + 情绪开关
    
    策略逻辑：
    1. 情绪过滤：涨停家数 >= 30
    2. 筛选条件：
       - 市值：50-150亿
       - 位置：<= 30%（相对历史最高点）
       - 无连板
    3. 假弱高开定义：
       - 开盘涨幅 > 0 但 < 3%
       - 开盘价 < 竞价最高价
    4. 卖出规则：
       - 次日冲高 +3% 即卖
       - 否则尾盘卖
    5. 停手机制：连亏3笔停3天
    """
    
    def __init__(self):
        self.name = "主线_假弱高开"
        self.market_cap_min = 50
        self.market_cap_max = 150
        self.position_limit = 30
        self.emotion_threshold = 30
        self.open_change_min = 0
        self.open_change_max = 0.03
        self.sell_profit_threshold = 0.03
        self.stop_loss_trades = 3
        self.stop_loss_days = 3
        
    def get_all_stocks(self, date):
        """
        获取所有A股股票列表
        Ricequant正确API: all_instruments(type='CS')
        """
        try:
            instruments = all_instruments(type='CS', date=date)
            return list(instruments.order_book_id)
        except:
            return []
        
    def get_limit_up_count(self, date):
        """
        获取涨停股票数量
        Ricequant正确API: history_bars()
        """
        all_stocks = self.get_all_stocks(date)[:500]
        limit_up_count = 0
        
        for stock in all_stocks:
            try:
                bars = history_bars(stock, 2, '1d', 'close', date)
                if bars is not None and len(bars) >= 2:
                    prev_close = bars[-2]
                    curr_close = bars[-1]
                    if prev_close > 0:
                        pct_change = (curr_close - prev_close) / prev_close
                        if pct_change >= 0.095:
                            limit_up_count += 1
            except:
                continue
        
        return limit_up_count
    
    def is_fake_weak_high_open(self, stock, date, prev_close):
        """
        判断是否为假弱高开
        Ricequant正确API: history_bars()
        """
        try:
            bars = history_bars(stock, 1, '1d', ['open', 'high'], date)
            if bars is None or len(bars) == 0:
                return False
            
            open_price = bars[-1]['open']
            high_price = bars[-1]['high']
            
            if prev_close > 0:
                open_change = (open_price - prev_close) / prev_close
            else:
                return False
            
            if open_change <= self.open_change_min or open_change >= self.open_change_max:
                return False
            
            if high_price > open_price:
                return True
            
            return False
        except:
            return False
    
    def get_stock_market_cap(self, stock, date):
        """
        获取股票市值（亿元）
        Ricequant正确API: get_factor()
        """
        try:
            factor_data = get_factor(stock, 'market_cap', start_date=date, end_date=date)
            if factor_data is not None and len(factor_data) > 0:
                return factor_data.iloc[0]['market_cap'] / 1e8
            return None
        except:
            return None
    
    def get_stock_position_pct(self, stock, date):
        """
        获取股票当前位置（相对历史最高点的百分比）
        Ricequant正确API: history_bars()
        """
        try:
            bars = history_bars(stock, 250, '1d', 'high', date)
            if bars is None or len(bars) == 0:
                return None
            
            highs = [bar['high'] for bar in bars]
            high_250d = max(highs)
            current_high = highs[-1]
            
            if high_250d > 0:
                position_pct = (current_high / high_250d - 1) * 100
                return position_pct
            return None
        except:
            return None
    
    def has_consecutive_boards(self, stock, date):
        """
        判断是否有连板
        Ricequant正确API: history_bars()
        """
        try:
            bars = history_bars(stock, 5, '1d', 'close', date)
            if bars is None or len(bars) < 5:
                return False
            
            closes = [bar['close'] for bar in bars]
            
            for i in range(len(closes) - 2):
                if closes[i] > 0 and closes[i+1] > 0:
                    pct_change = (closes[i+1] - closes[i]) / closes[i]
                    if pct_change >= 0.095:
                        return True
            
            return False
        except:
            return False
    
    def generate_signals(self, date):
        """
        生成信号
        """
        limit_up_count = self.get_limit_up_count(date)
        if limit_up_count < self.emotion_threshold:
            return []
        
        all_stocks = self.get_all_stocks(date)[:200]
        signals = []
        
        for stock in all_stocks:
            market_cap = self.get_stock_market_cap(stock, date)
            if market_cap is None or market_cap < self.market_cap_min or market_cap > self.market_cap_max:
                continue
            
            position_pct = self.get_stock_position_pct(stock, date)
            if position_pct is not None and position_pct > self.position_limit:
                continue
            
            if self.has_consecutive_boards(stock, date):
                continue
            
            try:
                bars = history_bars(stock, 2, '1d', 'close', date)
                if bars is None or len(bars) < 2:
                    continue
                prev_close = bars[-2]['close']
            except:
                continue
            
            if self.is_fake_weak_high_open(stock, date, prev_close):
                signals.append({
                    'stock': stock,
                    'date': date,
                    'signal_type': 'fake_weak_high_open',
                    'market_cap': market_cap,
                    'position_pct': position_pct
                })
        
        return signals

print("主线策略类定义完成")

## 2. 观察线策略：二板

In [ ]:
class SecondBoardStrategy:
    """
    观察线策略：二板策略
    
    策略逻辑：
    1. 只做二板（连板数量=2）
    2. 不做三板、四板
    3. 不做涨停排板
    4. 不接情绪层
    """
    
    def __init__(self):
        self.name = "观察线_二板"
        self.board_count = 2
        self.emotion_layer_enabled = False
        
    def get_all_stocks(self, date):
        """
        获取所有A股股票列表
        Ricequant正确API: all_instruments(type='CS')
        """
        try:
            instruments = all_instruments(type='CS', date=date)
            return list(instruments.order_book_id)
        except:
            return []
        
    def is_second_board(self, stock, date):
        """
        判断是否为二板
        Ricequant正确API: history_bars()
        """
        try:
            bars = history_bars(stock, 5, '1d', 'close', date)
            if bars is None or len(bars) < 5:
                return False
            
            closes = [bar['close'] for bar in bars]
            consecutive_boards = 0
            
            for i in range(len(closes) - 1):
                if closes[i] > 0:
                    pct_change = (closes[i+1] - closes[i]) / closes[i]
                    if pct_change >= 0.095:
                        consecutive_boards += 1
                    else:
                        break
            
            return consecutive_boards == 2
        except:
            return False
    
    def is_limit_up_today(self, stock, date):
        """
        判断今日是否涨停
        Ricequant正确API: history_bars()
        """
        try:
            bars = history_bars(stock, 2, '1d', 'close', date)
            if bars is None or len(bars) < 2:
                return False
            
            prev_close = bars[-2]['close']
            curr_close = bars[-1]['close']
            
            if prev_close > 0:
                pct_change = (curr_close - prev_close) / prev_close
                return pct_change >= 0.095
            
            return False
        except:
            return False
    
    def generate_signals(self, date):
        """
        生成信号
        """
        all_stocks = self.get_all_stocks(date)[:200]
        signals = []
        
        for stock in all_stocks:
            if not self.is_second_board(stock, date):
                continue
            
            if self.is_limit_up_today(stock, date):
                continue
            
            signals.append({
                'stock': stock,
                'date': date,
                'signal_type': 'second_board'
            })
        
        return signals

print("观察线策略类定义完成")

## 3. 回测引擎

In [ ]:
class BacktestEngine:
    """
    回测引擎
    Ricequant正确API: get_trading_dates(), history_bars()
    """
    
    def __init__(self, start_date, end_date, initial_capital=100000):
        self.start_date = start_date
        self.end_date = end_date
        self.initial_capital = initial_capital
        self.capital = initial_capital
        self.positions = {}
        self.trades = []
        self.daily_values = []
        self.consecutive_losses = 0
        self.stop_trading_until = None
        
    def get_trade_dates(self):
        """
        获取交易日列表
        Ricequant正确API: get_trading_dates()
        """
        try:
            dates = get_trading_dates(self.start_date, self.end_date)
            return list(dates)
        except:
            return pd.date_range(self.start_date, self.end_date, freq='B')
        
    def get_current_price(self, stock, date):
        """
        获取当前价格
        Ricequant正确API: history_bars()
        """
        try:
            bars = history_bars(stock, 1, '1d', 'close', date)
            if bars is not None and len(bars) > 0:
                return bars[-1]['close']
            return None
        except:
            return None
        
    def buy(self, stock, date, price, shares, reason=''):
        amount = price * shares
        if amount > self.capital:
            return False
        
        self.capital -= amount
        self.positions[stock] = {
            'buy_date': date,
            'buy_price': price,
            'shares': shares,
            'reason': reason
        }
        
        self.trades.append({
            'date': date,
            'stock': stock,
            'action': 'buy',
            'price': price,
            'shares': shares,
            'reason': reason
        })
        
        return True
    
    def sell(self, stock, date, price, reason=''):
        if stock not in self.positions:
            return False
        
        position = self.positions[stock]
        amount = price * position['shares']
        self.capital += amount
        
        profit = (price - position['buy_price']) * position['shares']
        profit_pct = (price - position['buy_price']) / position['buy_price']
        
        if profit < 0:
            self.consecutive_losses += 1
        else:
            self.consecutive_losses = 0
        
        self.trades.append({
            'date': date,
            'stock': stock,
            'action': 'sell',
            'price': price,
            'shares': position['shares'],
            'profit': profit,
            'profit_pct': profit_pct,
            'hold_days': (pd.to_datetime(date) - pd.to_datetime(position['buy_date'])).days,
            'reason': reason
        })
        
        del self.positions[stock]
        
        return True
    
    def calculate_total_value(self, date):
        total_value = self.capital
        
        for stock, position in self.positions.items():
            price = self.get_current_price(stock, date)
            if price is not None:
                total_value += price * position['shares']
        
        return total_value
    
    def run_backtest(self, strategy):
        trade_dates = self.get_trade_dates()
        
        print(f"回测时间范围: {self.start_date} 至 {self.end_date}")
        print(f"策略: {strategy.name}")
        print(f"初始资金: {self.initial_capital}")
        print(f"交易日数量: {len(trade_dates)}")
        print("-" * 80)
        
        for i, date in enumerate(trade_dates):
            if self.stop_trading_until is not None and date < self.stop_trading_until:
                continue
            else:
                self.stop_trading_until = None
            
            total_value = self.calculate_total_value(date)
            self.daily_values.append({
                'date': date,
                'value': total_value
            })
            
            stocks_to_sell = []
            for stock, position in self.positions.items():
                if hasattr(strategy, 'sell_profit_threshold'):
                    current_price = self.get_current_price(stock, date)
                    if current_price is not None:
                        profit_pct = (current_price - position['buy_price']) / position['buy_price']
                        if profit_pct >= strategy.sell_profit_threshold:
                            stocks_to_sell.append((stock, '冲高+3%止盈'))
                        elif (pd.to_datetime(date) - pd.to_datetime(position['buy_date'])).days >= 1:
                            stocks_to_sell.append((stock, '尾盘卖出'))
                else:
                    if (pd.to_datetime(date) - pd.to_datetime(position['buy_date'])).days >= 1:
                        stocks_to_sell.append((stock, '次日卖出'))
            
            for stock, reason in stocks_to_sell:
                price = self.get_current_price(stock, date)
                if price is not None:
                    self.sell(stock, date, price, reason)
            
            if self.consecutive_losses >= 3:
                self.stop_trading_until = pd.to_datetime(date) + timedelta(days=3)
                self.consecutive_losses = 0
                print(f"{date}: 连亏3笔，停手3天")
                continue
            
            signals = strategy.generate_signals(date)
            
            for signal in signals[:1]:
                stock = signal['stock']
                price = self.get_current_price(stock, date)
                if price is not None and self.capital > 0:
                    max_amount = min(100000, self.capital)
                    shares = int(max_amount / price / 100) * 100
                    if shares > 0:
                        self.buy(stock, date, price, shares, signal['signal_type'])
            
            if (i + 1) % 100 == 0:
                print(f"已处理 {i+1}/{len(trade_dates)} 个交易日")
        
        print("-" * 80)
        print("回测完成")
        
        return self.get_performance()
    
    def get_performance(self):
        if len(self.daily_values) == 0:
            return {}
        
        df = pd.DataFrame(self.daily_values)
        df['date'] = pd.to_datetime(df['date'])
        df.set_index('date', inplace=True)
        
        df['return'] = df['value'].pct_change()
        
        total_return = (df['value'].iloc[-1] - self.initial_capital) / self.initial_capital
        
        years = (df.index[-1] - df.index[0]).days / 365
        annual_return = (1 + total_return) ** (1 / years) - 1 if years > 0 else 0
        
        df['cummax'] = df['value'].cummax()
        df['drawdown'] = (df['value'] - df['cummax']) / df['cummax']
        max_drawdown = df['drawdown'].min()
        
        sharpe_ratio = df['return'].mean() / df['return'].std() * np.sqrt(252) if df['return'].std() > 0 else 0
        
        trades_df = pd.DataFrame(self.trades)
        if len(trades_df) > 0:
            sell_trades = trades_df[trades_df['action'] == 'sell']
            win_rate = len(sell_trades[sell_trades['profit'] > 0]) / len(sell_trades) if len(sell_trades) > 0 else 0
            avg_profit = sell_trades['profit'].mean() if len(sell_trades) > 0 else 0
            avg_profit_pct = sell_trades['profit_pct'].mean() if len(sell_trades) > 0 else 0
            avg_hold_days = sell_trades['hold_days'].mean() if len(sell_trades) > 0 else 0
        else:
            win_rate = 0
            avg_profit = 0
            avg_profit_pct = 0
            avg_hold_days = 0
        
        return {
            'total_return': total_return,
            'annual_return': annual_return,
            'max_drawdown': max_drawdown,
            'sharpe_ratio': sharpe_ratio,
            'win_rate': win_rate,
            'avg_profit': avg_profit,
            'avg_profit_pct': avg_profit_pct,
            'avg_hold_days': avg_hold_days,
            'trade_count': len(self.trades) // 2,
            'daily_values': df
        }

print("回测引擎定义完成")

## 4. 运行主线策略回测

In [ ]:
mainline_strategy = MainlineStrategy()
mainline_backtest = BacktestEngine('2014-01-01', '2024-12-31', initial_capital=100000)
mainline_performance = mainline_backtest.run_backtest(mainline_strategy)

print("\n" + "="*80)
print("主线策略回测结果")
print("="*80)
print(f"总收益率: {mainline_performance['total_return']:.2%}")
print(f"年化收益率: {mainline_performance['annual_return']:.2%}")
print(f"最大回撤: {mainline_performance['max_drawdown']:.2%}")
print(f"夏普比率: {mainline_performance['sharpe_ratio']:.2f}")
print(f"胜率: {mainline_performance['win_rate']:.2%}")
print(f"平均盈利: {mainline_performance['avg_profit']:.2f}")
print(f"平均盈利百分比: {mainline_performance['avg_profit_pct']:.2%}")
print(f"平均持仓天数: {mainline_performance['avg_hold_days']:.2f}")
print(f"交易次数: {mainline_performance['trade_count']}")

In [ ]:
if 'daily_values' in mainline_performance:
    plt.figure(figsize=(14, 6))
    plt.plot(mainline_performance['daily_values'].index, 
             mainline_performance['daily_values']['value'], 
             label='主线策略', linewidth=2)
    plt.axhline(y=100000, color='gray', linestyle='--', label='初始资金')
    plt.title('主线策略净值曲线 (2014-2024)', fontsize=16)
    plt.xlabel('日期', fontsize=12)
    plt.ylabel('资产净值', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 5. 运行观察线策略回测

In [ ]:
second_board_strategy = SecondBoardStrategy()
second_board_backtest = BacktestEngine('2014-01-01', '2024-12-31', initial_capital=100000)
second_board_performance = second_board_backtest.run_backtest(second_board_strategy)

print("\n" + "="*80)
print("观察线策略回测结果")
print("="*80)
print(f"总收益率: {second_board_performance['total_return']:.2%}")
print(f"年化收益率: {second_board_performance['annual_return']:.2%}")
print(f"最大回撤: {second_board_performance['max_drawdown']:.2%}")
print(f"夏普比率: {second_board_performance['sharpe_ratio']:.2f}")
print(f"胜率: {second_board_performance['win_rate']:.2%}")
print(f"平均盈利: {second_board_performance['avg_profit']:.2f}")
print(f"平均盈利百分比: {second_board_performance['avg_profit_pct']:.2%}")
print(f"平均持仓天数: {second_board_performance['avg_hold_days']:.2f}")
print(f"交易次数: {second_board_performance['trade_count']}")

In [ ]:
if 'daily_values' in second_board_performance:
    plt.figure(figsize=(14, 6))
    plt.plot(second_board_performance['daily_values'].index, 
             second_board_performance['daily_values']['value'], 
             label='观察线策略', linewidth=2)
    plt.axhline(y=100000, color='gray', linestyle='--', label='初始资金')
    plt.title('观察线策略净值曲线 (2014-2024)', fontsize=16)
    plt.xlabel('日期', fontsize=12)
    plt.ylabel('资产净值', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 6. 策略对比

In [ ]:
comparison = pd.DataFrame({
    '主线策略': [
        f"{mainline_performance['total_return']:.2%}",
        f"{mainline_performance['annual_return']:.2%}",
        f"{mainline_performance['max_drawdown']:.2%}",
        f"{mainline_performance['sharpe_ratio']:.2f}",
        f"{mainline_performance['win_rate']:.2%}",
        f"{mainline_performance['trade_count']}
    ],
    '观察线策略': [
        f"{second_board_performance['total_return']:.2%}",
        f"{second_board_performance['annual_return']:.2%}",
        f"{second_board_performance['max_drawdown']:.2%}",
        f"{second_board_performance['sharpe_ratio']:.2f}",
        f"{second_board_performance['win_rate']:.2%}",
        f"{second_board_performance['trade_count']}
    ]
}, index=['总收益率', '年化收益率', '最大回撤', '夏普比率', '胜率', '交易次数'])

print("\n" + "="*80)
print("策略对比")
print("="*80)
print(comparison)

In [ ]:
if 'daily_values' in mainline_performance and 'daily_values' in second_board_performance:
    plt.figure(figsize=(14, 8))
    
    plt.subplot(2, 1, 1)
    plt.plot(mainline_performance['daily_values'].index, 
             mainline_performance['daily_values']['value'], 
             label='主线策略', linewidth=2)
    plt.plot(second_board_performance['daily_values'].index, 
             second_board_performance['daily_values']['value'], 
             label='观察线策略', linewidth=2)
    plt.axhline(y=100000, color='gray', linestyle='--', label='初始资金')
    plt.title('策略净值对比 (2014-2024)', fontsize=16)
    plt.xlabel('日期', fontsize=12)
    plt.ylabel('资产净值', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 1, 2)
    plt.plot(mainline_performance['daily_values'].index, 
             mainline_performance['daily_values']['drawdown'], 
             label='主线策略', linewidth=2)
    plt.plot(second_board_performance['daily_values'].index, 
             second_board_performance['daily_values']['drawdown'], 
             label='观察线策略', linewidth=2)
    plt.title('策略回撤对比', fontsize=16)
    plt.xlabel('日期', fontsize=12)
    plt.ylabel('回撤', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 7. 总结

In [ ]:
print("\n" + "="*80)
print("回测总结")
print("="*80)

print("\n主线策略（假弱高开）:")
print("- 情绪过滤：涨停家数 >= 30")
print("- 筛选：市值50-150亿 + 位置<=30% + 无连板")
print("- 卖出：冲高+3%止盈或尾盘卖出")
print(f"- 10年总收益: {mainline_performance['total_return']:.2%}")
print(f"- 年化收益: {mainline_performance['annual_return']:.2%}")
print(f"- 最大回撤: {mainline_performance['max_drawdown']:.2%}")
print(f"- 夏普比率: {mainline_performance['sharpe_ratio']:.2f}")
print(f"- 胜率: {mainline_performance['win_rate']:.2%}")

print("\n观察线策略（二板）:")
print("- 只做二板（不做三板、四板、涨停排板）")
print("- 不接情绪层")
print(f"- 10年总收益: {second_board_performance['total_return']:.2%}")
print(f"- 年化收益: {second_board_performance['annual_return']:.2%}")
print(f"- 最大回撤: {second_board_performance['max_drawdown']:.2%}")
print(f"- 夏普比率: {second_board_performance['sharpe_ratio']:.2f}")
print(f"- 胜率: {second_board_performance['win_rate']:.2%}")

print("\n建议:")
print("1. 主线策略信号较少，适合极小仓验证")
print("2. 观察线策略表现稳定，可逐步加仓")
print("3. 两策略可并行运行，分散风险")
print("4. 建议单票上限10万，总仓上限30万")